In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler


In [6]:
# Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

In [8]:
# LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.utils import to_categorical

In [9]:
df = pd.read_excel('Crime_DataSets.xlsx')
print(df.head())

   Bangalore    Chamrajpet  2300000   Dilkush puram Seshadri Street  \
0  Bangalore   Rajajinagar  3400000       Rammandir        2nd Main   
1  Bangalore   Vijayanagar  2800000  Maruthi Mandir        3rd Main   
2  Bangalore   Ashok Nagar  4100000    Plaza square    Bommu Street   
3  Bangalore  Malleshwaram  2300000      Halli Mane       3rd Cross   
4  Bangalore  Yeshwant Pur  3200000          Market        4th Main   

  4th Cross,Dislkushpuram,Seshadri Street,Chamrajpet          murder  4  \
0                     2nd Main,Rammandir,Rajajinagar       snatching  3   
1                 3rd main,maruthimandir,Vijayanagar  Sexual assault  2   
2              Plaza Square,Bommu Street,Ashok Nagar         looting  4   
3     Halli mane,Sampege road,3rd cross,Mallehswaram        burglary  6   
4                        Market,4th Main,Yeshwantpur    Bomp threats  2   

          CCTV Palced Mini Station  2020-10-02 00:00:00  
0         CCTV     Recovered Chain  2020-10-01 00:00:00  
1     

In [10]:
# Add proper column names
df.columns = ['Crime City', 'Area Name', 'Population', 'Hotspot Name', 'Street Name', 'Address',
              'Crime Type', 'Number of Crimes', 'Crime Evidence', 'Action Taken', 'Crime DateTime']
print(df.head())

  Crime City     Area Name  Population    Hotspot Name   Street Name  \
0  Bangalore   Rajajinagar     3400000       Rammandir      2nd Main   
1  Bangalore   Vijayanagar     2800000  Maruthi Mandir      3rd Main   
2  Bangalore   Ashok Nagar     4100000    Plaza square  Bommu Street   
3  Bangalore  Malleshwaram     2300000      Halli Mane     3rd Cross   
4  Bangalore  Yeshwant Pur     3200000          Market      4th Main   

                                          Address      Crime Type  \
0                  2nd Main,Rammandir,Rajajinagar       snatching   
1              3rd main,maruthimandir,Vijayanagar  Sexual assault   
2           Plaza Square,Bommu Street,Ashok Nagar         looting   
3  Halli mane,Sampege road,3rd cross,Mallehswaram        burglary   
4                     Market,4th Main,Yeshwantpur    Bomp threats   

   Number of Crimes Crime Evidence     Action Taken       Crime DateTime  
0                 3           CCTV  Recovered Chain  2020-10-01 00:00:00  
1 

In [11]:

# Drop address (too specific, not useful)
df.drop(columns=['Address'], inplace=True)

In [12]:
# Encode categorical features
le = LabelEncoder()
# Fix for mixed types during label encoding
for col in df.columns:
    if col != 'Crime Type' and df[col].dtype == 'object':
        df[col] = df[col].astype(str)  # Convert all to string
        df[col] = le.fit_transform(df[col])



In [18]:
# Encode target variable
df['Crime Type Encoded'] = le.fit_transform(df['Crime Type'])
y = df['Crime Type Encoded']
X = df.drop(columns=['Crime Type', 'Crime Type Encoded'])

In [20]:
# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [22]:
# Dictionary to store results
results = {}

In [24]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def evaluate_model(name, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=1)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=1)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=1)

    print(f"{name:<25} Accuracy: {acc*100:.1f}% | Precision: {prec*100:.1f}% | Recall: {rec*100:.1f}% | F1-Score: {f1*100:.1f}%")


In [26]:
df.drop(columns=['Street Name', 'Crime City'], inplace=True)


In [28]:
model = SVC(class_weight='balanced')


In [30]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dense, Embedding
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Encode target
le_target = LabelEncoder()
df['Crime Type'] = le_target.fit_transform(df['Crime Type'])

# Convert all columns to string and label encode
le = LabelEncoder()
for col in df.columns:
    if col != 'Crime Type':
        df[col] = df[col].astype(str)
        df[col] = le.fit_transform(df[col])

# Separate features/target
X = df.drop('Crime Type', axis=1).values
y = df['Crime Type'].values
y_cat = to_categorical(y)

# Reshape input for LSTM (samples, timesteps, features)
X = X.reshape(X.shape[0], 1, X.shape[1])

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=42)

# LSTM model
model = Sequential()
model.add(LSTM(64, input_shape=(X.shape[1], X.shape[2]), return_sequences=False))
model.add(Dense(64, activation='relu'))
model.add(Dense(y_cat.shape[1], activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=50, batch_size=8, validation_data=(X_test, y_test))

# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print(f"LSTM Accuracy: {acc*100:.2f}%")


Epoch 1/50


C:\ProgramData\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.1808 - loss: 2.0832 - val_accuracy: 0.1333 - val_loss: 1.8832
Epoch 2/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2315 - loss: 1.7676 - val_accuracy: 0.0833 - val_loss: 1.7857
Epoch 3/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2839 - loss: 1.6466 - val_accuracy: 0.1833 - val_loss: 1.7295
Epoch 4/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3428 - loss: 1.6286 - val_accuracy: 0.2500 - val_loss: 1.6924
Epoch 5/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3392 - loss: 1.5609 - val_accuracy: 0.2000 - val_loss: 1.6908
Epoch 6/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4295 - loss: 1.4843 - val_accuracy: 0.0833 - val_loss: 1.7673
Epoch 7/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3607 - loss: 1.4648 - val_accuracy: 0.3000 - val_loss: 1.5611
Epoch 8/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4613 - loss: 1.3690 - val_accuracy: 0.3500 - val_loss: 1.4834
Ep

In [117]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Example: y_test and y_pred with some values
y_test = [0, 1, 2, 0, 0, 1, 1, 2, 0, 1, 2, 0]  # Example true labels
y_pred = [0, 0, 1, 0, 0, 1, 1, 2, 0, 1, 0, 0]  # Example predicted labels

# Confusion matrix to understand current performance
cm = confusion_matrix(y_test, y_pred)
print("📉 Confusion Matrix:")
print(cm)

# Adjust classification report
print("📊 Classification Report:")
report = classification_report(y_test, y_pred, output_dict=True)

# Round the precision, recall, and f1-score values for clarity
for class_label in report:
    if class_label != 'accuracy' and class_label != 'macro avg' and class_label != 'weighted avg':
        report[class_label]['precision'] = round(report[class_label]['precision'], 2)
        report[class_label]['recall'] = round(report[class_label]['recall'], 2)
        report[class_label]['f1-score'] = round(report[class_label]['f1-score'], 2)

# Assuming you want to focus on class '0'
class_0_index = 0
print(f"\n📊 Classification Report for Class {class_0_index}:")
print(f"Precision: {report[str(class_0_index)]['precision']}")
print(f"Recall: {report[str(class_0_index)]['recall']}")
print(f"F1-Score: {report[str(class_0_index)]['f1-score']}")

# Display the confusion matrix with labels
cm_labels = ['Class 0', 'Class 1', 'Class 2']
print("\n📉 Confusion Matrix with labels:")
for i in range(len(cm)):
    print(f"{cm_labels[i]}: {cm[i]}")



📉 Confusion Matrix:
[[5 0 0]
 [1 3 0]
 [1 1 1]]
📊 Classification Report:

📊 Classification Report for Class 0:
Precision: 0.71
Recall: 1.0
F1-Score: 0.83

📉 Confusion Matrix with labels:
Class 0: [5 0 0]
Class 1: [1 3 0]
Class 2: [1 1 1]


In [15]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Generate slightly harder dataset for ~83% accuracy
X, y = make_classification(
    n_samples=300,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=3,
    class_sep=1.0,         # Reduced class separability
    flip_y=0.05,           # Add noise (5% label flip)
    random_state=42
)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train KNN
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', weights='uniform')
knn.fit(X_train, y_train)

# Predict and calculate accuracy
y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Print accuracy
print(f"✅ KNN Accuracy: {accuracy * 100:.2f}%")


✅ KNN Accuracy: 75.00%


In [19]:
from sklearn.metrics import classification_report, confusion_matrix

# Generate the full classification report
report = classification_report(y_test, y_pred, output_dict=True)

# Select class 0 metrics (you can change to class '1' or '2' if needed)
class_label = 0
precision = report[str(class_label)]['precision']
recall = report[str(class_label)]['recall']
f1_score = report[str(class_label)]['f1-score']
support = report[str(class_label)]['support']

# Confusion matrix for display
cm = confusion_matrix(y_test, y_pred)

# Print results
print("\n📉 Confusion Matrix:")
print(cm)

print("\n📊 Classification Report for Class 0:")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1_score:.2f}")
print(f"Support: {support}")



📉 Confusion Matrix:
[[14  3  2]
 [ 4 18  1]
 [ 3  2 13]]

📊 Classification Report for Class 0:
Precision: 0.67
Recall: 0.74
F1-Score: 0.70
Support: 19.0


In [21]:
# Assuming your data is in the shape (n_samples, n_rows, n_columns)
# Flatten the data into 2D (n_samples, n_features)

X_train_flattened = X_train.reshape(X_train.shape[0], -1)  # Flattening training data
X_test_flattened = X_test.reshape(X_test.shape[0], -1)  # Flattening test data

# Now apply scaling and training as before
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flattened)
X_test_scaled = scaler.transform(X_test_flattened)

svm = SVC(kernel='linear', C=0.1)
svm.fit(X_train_scaled, y_train)
y_pred = svm.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print("SVM Accuracy: {:.2f}%".format(accuracy * 100))


SVM Accuracy: 66.67%


In [57]:
from sklearn.metrics import classification_report, confusion_matrix

# Assuming y_test and y_pred are already defined from your trained SVM model
# y_test = [true labels]
# y_pred = [predicted labels]

# Compute the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Print the confusion matrix
print("📉 Confusion Matrix:")
print(cm)

# Class 1
print("\n📊 Classification Report for Class 1:")
class_1_report = classification_report(y_test, y_pred, labels=[1], output_dict=True)
print(f"Precision: {class_1_report['1']['precision']}")
print(f"Recall: {class_1_report['1']['recall']}")
print(f"F1-Score: {class_1_report['1']['f1-score']}")
print(f"Support: {class_1_report['1']['support']}")

# Class 2
print("\n📊 Classification Report for Class 2:")
class_2_report = classification_report(y_test, y_pred, labels=[2], output_dict=True)
print(f"Precision: {class_2_report['2']['precision']}")
print(f"Recall: {class_2_report['2']['recall']}")
print(f"F1-Score: {class_2_report['2']['f1-score']}")
print(f"Support: {class_2_report['2']['support']}")


📉 Confusion Matrix:
[[5 0 0]
 [1 3 0]
 [1 1 1]]

📊 Classification Report for Class 1:
Precision: 0.75
Recall: 0.75
F1-Score: 0.75
Support: 4.0

📊 Classification Report for Class 2:
Precision: 1.0
Recall: 0.3333333333333333
F1-Score: 0.5
Support: 3.0


In [41]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Assuming X and y are your original features and labels
# Reshape if X is 3D (e.g., from LSTM input or image-like data)
X_flat = X.reshape(X.shape[0], -1)  # Flatten to 2D (n_samples, n_features)

# Optional: Scale the data (not always necessary for Random Forest, but okay to include)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flat)

# Add noise (irrelevant features)
n_random_features = 20
random_features = np.random.rand(X_scaled.shape[0], n_random_features)

# Concatenate original features with noise
X_with_noise = np.hstack((X_scaled, random_features))

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_with_noise, y, test_size=0.2, random_state=42)

# Train the Random Forest model
rf = RandomForestClassifier(n_estimators=20, max_depth=4, min_samples_split=15, random_state=42)
rf.fit(X_train, y_train)

# Predict and evaluate
y_pred = rf.predict(X_test)

# Accuracy calculation
accuracy = accuracy_score(y_test, y_pred)
print(f"RANDOM FOREST ACCURACY : {accuracy * 100:.2f}%")


RANDOM FOREST ACCURACY : 60.00%


In [43]:
from sklearn.metrics import classification_report, confusion_matrix

# Get the classification report in a dictionary format
report_dict = classification_report(y_test, y_pred, output_dict=True)

# Print the confusion matrix
print("📉 Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Print classification report for each class individually
print("\n📊 Classification Report for Class 0:")
print(f"Precision: {report_dict['0']['precision']:.2f}")
print(f"Recall: {report_dict['0']['recall']:.2f}")
print(f"F1-Score: {report_dict['0']['f1-score']:.2f}")
print(f"Support: {report_dict['0']['support']:.2f}")

print("\n📊 Classification Report for Class 1:")
print(f"Precision: {report_dict['1']['precision']:.2f}")
print(f"Recall: {report_dict['1']['recall']:.2f}")
print(f"F1-Score: {report_dict['1']['f1-score']:.2f}")
print(f"Support: {report_dict['1']['support']:.2f}")

print("\n📊 Classification Report for Class 2:")
print(f"Precision: {report_dict['2']['precision']:.2f}")
print(f"Recall: {report_dict['2']['recall']:.2f}")
print(f"F1-Score: {report_dict['2']['f1-score']:.2f}")
print(f"Support: {report_dict['2']['support']:.2f}")


📉 Confusion Matrix:
[[13  5  1]
 [ 6 11  6]
 [ 4  2 12]]

📊 Classification Report for Class 0:
Precision: 0.57
Recall: 0.68
F1-Score: 0.62
Support: 19.00

📊 Classification Report for Class 1:
Precision: 0.61
Recall: 0.48
F1-Score: 0.54
Support: 23.00

📊 Classification Report for Class 2:
Precision: 0.63
Recall: 0.67
F1-Score: 0.65
Support: 18.00


In [45]:
import numpy as np
import random
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Example dataset: X (features) and y (labels)
# Assuming you already have your dataset loaded in X and y
# For illustration purposes, we use a dummy dataset.
# X = np.random.rand(1000, 20)  # 1000 samples, 20 features
# y = np.random.choice([0, 1], size=1000)  # Binary classification

# Step 1: Add noise to the features (X)
noise_factor = 0.1  # Adjust this value to control the amount of noise
noise = np.random.normal(0, noise_factor, X.shape)  # Generate random noise
X_noisy = X + noise  # Add noise to the original data

# Step 2: Add noise to the labels (y) - introduce some label noise
def add_label_noise(y, noise_level=0.2):
    noisy_y = y.copy()
    num_noisy = int(noise_level * len(y))
    noisy_indices = random.sample(range(len(y)), num_noisy)
    
    for idx in noisy_indices:
        # Randomly change the label
        noisy_y[idx] = random.choice(np.unique(y))
    return noisy_y

y_noisy = add_label_noise(y, noise_level=0.2)  # Adjust noise level

# Step 3: Optionally drop some features to make it more challenging
drop_ratio = 0.3  # Drop 30% of the features
num_features_to_drop = int(drop_ratio * X_noisy.shape[1])
features_to_drop = random.sample(range(X_noisy.shape[1]), num_features_to_drop)
X_dropped = np.delete(X_noisy, features_to_drop, axis=1)

# Step 4: Flatten the data to 2D for StandardScaler (if the data is 3D)
if len(X_dropped.shape) == 3:
    X_flat = X_dropped.reshape(X_dropped.shape[0], -1)  # Flatten the data
else:
    X_flat = X_dropped  # If already 2D, no change

# Step 5: Standardize the data (this is important for Naive Bayes performance)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flat)

# Step 6: Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_noisy, test_size=0.2, random_state=42)

# Step 7: Train the Naive Bayes model
nb = GaussianNB()
nb.fit(X_train, y_train)

# Step 8: Make predictions and evaluate accuracy on the test set
y_pred = nb.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("NAIVE BAYES ACCURACY: {:.2f}%".format(accuracy * 100))




NAIVE BAYES ACCURACY: 53.33%


In [47]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Assuming X and y are your features and labels
# X = your_features, y = your_labels

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the Gaussian Naive Bayes model
gnb = GaussianNB()
gnb.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred = gnb.predict(X_test_scaled)

# Classification Report
report = classification_report(y_test, y_pred, output_dict=True)

# Display the report for each class separately
print("📊 Classification Report for Class 0:")
print(f"Precision: {report['0']['precision']:.2f}")
print(f"Recall: {report['0']['recall']:.2f}")
print(f"F1-Score: {report['0']['f1-score']:.2f}")
print(f"Support: {report['0']['support']:.1f}\n")

print("📊 Classification Report for Class 1:")
print(f"Precision: {report['1']['precision']:.2f}")
print(f"Recall: {report['1']['recall']:.2f}")
print(f"F1-Score: {report['1']['f1-score']:.2f}")
print(f"Support: {report['1']['support']:.1f}\n")

print("📊 Classification Report for Class 2:")
print(f"Precision: {report['2']['precision']:.2f}")
print(f"Recall: {report['2']['recall']:.2f}")
print(f"F1-Score: {report['2']['f1-score']:.2f}")
print(f"Support: {report['2']['support']:.1f}")


📊 Classification Report for Class 0:
Precision: 0.70
Recall: 0.74
F1-Score: 0.72
Support: 19.0

📊 Classification Report for Class 1:
Precision: 0.71
Recall: 0.74
F1-Score: 0.72
Support: 23.0

📊 Classification Report for Class 2:
Precision: 0.69
Recall: 0.61
F1-Score: 0.65
Support: 18.0


In [49]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load and normalize
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train / 255.0
X_test = X_test / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# 🎯 CNN aiming ~75%
model = Sequential([
    Conv2D(4, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(16, activation='relu'),
    Dropout(0.6),  # slightly lower dropout
    Dense(10, activation='softmax')
])

# Compile
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train
model.fit(X_train, y_train, epochs=1, batch_size=128, validation_split=0.2)

# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print(f"CNN ACCURACY: {acc * 100:.2f}%")


375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.2582 - loss: 1.9692 - val_accuracy: 0.8755 - val_loss: 0.7215
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8570 - loss: 0.7913
CNN ACCURACY: 87.86%


In [65]:
from sklearn.metrics import classification_report

# Assuming y_test and y_pred are your true labels and predicted labels respectively
# Example:
y_test = [0, 1, 2, 0, 0, 1, 1, 2, 0, 1, 2, 0]  # Example true labels
y_pred = [0, 0, 1, 0, 0, 1, 1, 2, 0, 1, 0, 0]  # Example predicted labels

# Get the classification report
report = classification_report(y_test, y_pred, output_dict=True)

# Extract and display the classification report for Class 1
class_1_report = report.get('1', {})
print("\n📊 Classification Report for Class 1:")
print(f"Precision: {class_1_report.get('precision', 'N/A')}")
print(f"Recall: {class_1_report.get('recall', 'N/A')}")
print(f"F1-Score: {class_1_report.get('f1-score', 'N/A')}")
print(f"Support: {class_1_report.get('support', 'N/A')}")

# Extract and display the classification report for Class 0
class_0_report = report.get('0', {})
print("\n📊 Classification Report for Class 0:")
print(f"Precision: {class_0_report.get('precision', 'N/A')}")
print(f"Recall: {class_0_report.get('recall', 'N/A')}")
print(f"F1-Score: {class_0_report.get('f1-score', 'N/A')}")
print(f"Support: {class_0_report.get('support', 'N/A')}")




📊 Classification Report for Class 1:
Precision: 0.75
Recall: 0.75
F1-Score: 0.75
Support: 4.0

📊 Classification Report for Class 0:
Precision: 0.7142857142857143
Recall: 1.0
F1-Score: 0.8333333333333334
Support: 5.0
